# DATA VISUALIZATION(Spotify streaming)

### DATA SET IMPORTING

In [ ]:
import pandas as pd
df=pd.read_csv("Spotify_streaming_optimized.csv")
print(df)

# QUICK CHECK

In [ ]:
df.head()

In [ ]:
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns

### DATA CLEANING

### check for missing values

In [ ]:
df.isnull().sum()

### Remove unnecessary columns

In [ ]:
df.drop_duplicates(inplace=True)

df = df.drop(columns=["Unnamed: 0"], errors="ignore")
print(df.head())

### Clean text columns

Some columns contain text, but the text may have extra spaces, irregular formatting, or hidden characters.

Cleaning text columns makes the data neat, uniform and readable.
This helps avoid errors in groupby, pivot, and visualization.

In [ ]:
text_cols = ["Artist","Track","Album","Channel","Title"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    print(df[text_cols].head(1))

### Convert numerical columns

We convert them into numeric format so we can do groupby, sum, mean, plots, etc

In [ ]:
numeric_cols = ["Views","Likes","Comments","Stream","Danceability","Energy",
                "Key","Loudness","Speechiness","Acousticness","Instrumentalness",
                "Liveness","Valence","Tempo","Duration_ms"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].head(1))

### Handle missing values

In [ ]:
yt_num = ["Views","Likes","Comments","Stream"]
df[yt_num] = df[yt_num].fillna(0)

# Fill audio feature missing values with mean
audio_cols = ["Danceability","Energy","Key","Loudness","Speechiness"]
for col in audio_cols:
    df[col] = df[col].fillna(df[col].mean())

# Fill categorical missing values
cat_cols = ["Url_youtube","Title","Channel","Description"]
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Boolean fields
df["Licensed"] = df["Licensed"].fillna(False)
df["official_video"] = df["official_video"].fillna(False)

missing_audio_cols = ["Acousticness","Instrumentalness","Liveness","Valence","Tempo","Duration_ms"]

for col in missing_audio_cols:
    df[col] = df[col].fillna(df[col].mean())

print("📌 Missing values handled successfully!")

### Remove duplicates

In [ ]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]

print(f" Removed {before - after} duplicate rows")

In [ ]:
print(df.info())

In [ ]:
df.isnull().sum()

In [ ]:
df.nunique()

### Outlier Detection

Box plots help identify minimum, maximum, quartiles, and unusual values.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15,10))
num_cols = ["Views", "Comments", "Stream"]
for i, col in enumerate(num_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(data=df, y=col)
    plt.title(f"Boxplot of {col}")

plt.tight_layout()
plt.show()

## GroupBy 1: Artist vs Total Spotify Streams

### Objective

To identify which artists have the highest total streaming counts on Spotify and analyze their popularity.

In [ ]:
df.groupby("Artist")["Stream"].sum()

In [ ]:
g1 = df.groupby("Artist")["Stream"].sum().sort_values(ascending=False).head(10)
print(g1)

plt.figure(figsize=(10,6))
g1.plot(kind='bar')
plt.title("Top Artists by Total Youtube Streams")
plt.xlabel("Artist")
plt.ylabel("Total Streams")
plt.xticks(rotation=45)
plt.show()

# Inference

The bar graph shows the top artists who have the highest total Youtube streams.

Artists at the top of the chart have significantly larger fan bases and more popular tracks.

This analysis helps understand which artists dominate Youtube streaming trends.

# GroupBy 2:Artist vs Total Spotify streaming (Pie Chart)

## Objective

To find which artists generate the highest total YouTube views and represent their contribution using a PIE chart.

In [ ]:
df.groupby("Artist")["Views"].sum()

In [ ]:
g2 = df.groupby("Artist")["Views"].sum().sort_values(ascending=False).head(7)
print(g2)

plt.figure(figsize=(9,9))
plt.pie(g2, labels=g2.index, autopct="%1.1f%%", startangle=140)
plt.title("Top Artists by Spotify streaming (Pie Chart)")
plt.show()

# Inference

The pie chart shows each top artist’s percentage contribution to total YouTube views.

Some artists dominate a large portion of total views, meaning their music videos attract more audience attention.

Pie chart clearly reveals the proportion of views instead of absolute numbers.

# GroupBy 3: Channel vs Likes (Bubble Chart)

## Objective

To understand how different Spotify streaming perform in terms of likes using bubble sizes to represent engagement.

In [ ]:
df.groupby("Channel")["Likes"].sum()

In [ ]:
g3 = df.groupby("Channel")["Likes"].sum().sort_values(ascending=False).head(10)
print(g3)

plt.figure(figsize=(12,7))

plt.scatter(
    x=g3.index,
    y=g3.values,
    s=g3.values / 5000,   # bubble size
    alpha=0.6,
    edgecolors="black"
)

plt.xticks(rotation=45)
plt.title("Total Likes per Channel (Bubble Chart)")
plt.xlabel("Channel")
plt.ylabel("Total Likes")
plt.show()

# Inference

Larger bubbles indicate channels that have accumulated more likes.

This chart makes it easy to compare both value and importance visually.

Bubble charts are excellent for showing “magnitude” differences in engagement.

The most popular channels clearly stand out with larger bubble sizes.

# PIVOT 1: Artist vs Album Type (Energy Levels)

## Objective

To understand how energy levels change for different artists across album types using a pivot table and heatmap.

In [ ]:
pd.pivot_table(df, values="Energy", index="Artist", columns="Album_type",aggfunc="mean")

In [ ]:
p1 = pd.pivot_table(df, values="Energy", index="Artist", columns="Album_type", aggfunc="mean")
print(p1.head())

plt.figure(figsize=(12,8))
sns.heatmap(p1, cmap="YlOrRd")
plt.title("Energy Levels Across Artists and Album Types (Heatmap)")
plt.show()

# Inference

The heatmap clearly shows which artists produce more energetic songs.

Useful for identifying the music style of different artists

# PIVOT 2: Licensed vs Unlicensed Videos (YouTube Views)

## Objective

To compare the total YouTube views of licensed and non-licensed music videos across artists.

In [ ]:
pd.pivot_table(df, values="Views", index="Artist", columns="Licensed",aggfunc="sum")

In [ ]:
p2= pd.pivot_table(df, values="Views", index="Artist", columns="Licensed", aggfunc="sum")
print(p2.head())
p2_top = p2.head(10)
plt.figure(figsize=(12,7))
p2_top.plot(kind="bar", stacked=True)
plt.title("Licensed vs Unlicensed YouTube Views (Stacked Bar Chart)")
plt.ylabel("Total Views")
plt.xticks(rotation=45)
plt.show()

# Inference

The stacked bars show how views are distributed between licensed and unlicensed videos.

For some artists, licensed videos contribute the majority of views.

The chart helps observe artist performance based on content rights.

Stacked bar chart gives a clear comparison across categories.

# PIVOT 3: Channel vs Official Video (Likes)

## Objective

To analyze the average likes on official vs unofficial YouTube videos across channels.

In [ ]:
pd.pivot_table(df, values="Likes", index="Channel", columns="official_video",aggfunc="mean")

In [ ]:
p3 = pd.pivot_table(df, values="Likes", index="Channel", columns="official_video", aggfunc="mean")
print(p3.head())

p3_top = p3.head(10)

plt.figure(figsize=(12,7))
p3_top.plot(kind="area", alpha=0.5)
plt.title("Average Likes: Official vs Unofficial Videos (Area Chart)")
plt.ylabel("Average Likes")
plt.xticks(rotation=45)
plt.show()